<br>

# PyTorch 基础 2 

<br>

## 0. 概述

<br>

<font color=black size=3 face=雅黑>　　在上一次实验中，我们围绕 PyTorch 的基本数据类型 Tensor 做过一系列练习。本次 PyTorch 基础练习，我们将学习如下内容：

<font color=black size=3 face=雅黑>　　　(1) 在 GPU 上实现模型训练与测试，并与 CPU 上的训练时长对比；

<font color=black size=3 face=雅黑>　　　(2) 进一步了解 PyTorch 的自动求导功能。

<br>

## 1. GPU vs. CPU

<br>

<font color=black size=3 face=雅黑>　　在这一部分，我们将学习如何将数据和模型从 CPU 转移到 GPU 上，并在 GPU 中进行训练与测试。

<br>

### 1.1 数据集准备

<br>

<font color=black size=3 face=雅黑>　　我们首先来准备数据集 CIFAR-10。如果数据集在原网上下载太慢，大家可以像上节课一样，自行从实验材料中下载压缩包，创建路径“./dataset_cifar10/”并将压缩包上传。

<code>
%%html
<img src = "https://img-blog.csdnimg.cn/20200816150537995.png?x-oss-process=image/watermark,type_ZmFuZ3poZW5naGVpdGk,shadow_10,text_aHR0cHM6Ly9ibG9nLmNzZG4ubmV0L3dlaXhpbl80Mjg5OTYyNw==,size_16,color_FFFFFF,t_70#pic_center", width = 45%>
    
<br>

In [1]:
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
import time

<br>

<font color=black size=3 face=雅黑>与之前的实验相同，我们用 torchvision 读入数据，并对数据进行标准化处理，最后加载到 DataLoader 中。
    
<br>

In [2]:
batch_size = 200  # 设置训练集和测试集的 batch size，即每批次将参与运算的样本数

# 训练集
train_set = torchvision.datasets.CIFAR10(root='./dataset_cifar10', train=True, download=True,
                                        transform=torchvision.transforms.Compose([
                                            torchvision.transforms.ToTensor(), 
                                            torchvision.transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))]))
# 测试集
test_set = torchvision.datasets.CIFAR10(root='./dataset_cifar10', train=False, download=True,
                                        transform=torchvision.transforms.Compose([
                                            torchvision.transforms.ToTensor(), 
                                            torchvision.transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))]))

train_loader = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=batch_size, shuffle=True)

### 1.2 构建卷积神经网络

<br>

<font color=black size=3 face=雅黑>此处我们构建一个和上节课相同的卷积神经网络。

<br>

In [3]:
class Network(nn.Module):
    def __init__(self):
        super(Network, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=6, kernel_size=5)
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=12, kernel_size=5)
        
        self.fc1 = nn.Linear(in_features=12*5*5, out_features=120)
        self.fc2 = nn.Linear(in_features=120, out_features=60)
        self.out = nn.Linear(in_features=60, out_features=10)
        
        
    def forward(self, t):
        
        # conv1
        t = self.conv1(t)
        t = F.relu(t) 
        t = F.max_pool2d(t, kernel_size=2, stride=2) 
        
        # conv2
        t = self.conv2(t)
        t = F.relu(t)
        t = F.max_pool2d(t, kernel_size=2, stride=2)
        
        t = t.reshape(batch_size, 12*5*5)
        
        # fc1
        t = self.fc1(t)
        t = F.relu(t)
        
        # fc2
        t = self.fc2(t)
        t = F.relu(t)
        
        # output layer
        t = self.out(t)
        
        return t

<br>

### 1.3 在 GPU 上训练

<br>

In [4]:
network1 = Network()
network1.cuda()  # 将模型转移到 GPU 上

Network(
  (conv1): Conv2d(3, 6, kernel_size=(5, 5), stride=(1, 1))
  (conv2): Conv2d(6, 12, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=300, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=60, bias=True)
  (out): Linear(in_features=60, out_features=10, bias=True)
)

In [5]:
loss_func = nn.CrossEntropyLoss()  # 损失函数：交叉熵损失
optimizer1 = torch.optim.SGD(network1.parameters(), lr=0.1)  # 优化器

def get_num_correct(preds, labels):  # 计算正确分类的次数
    return preds.argmax(dim=1).eq(labels).sum().item()

In [6]:
total_epochs = 5
time_start1 = time.time()

for epoch in range(total_epochs):

    total_loss = 0
    total_train_correct = 0

    for batch in train_loader:  # 抓取一个 batch
        
        # 读取样本数据        
        images, labels = batch
        images = images.cuda()  # 数据转移到 GPU 上
        labels = labels.cuda()  # 标签转移到 GPU 上
        
        # 完成正向传播，计算损失
        preds = network1(images)
        loss = loss_func(preds, labels)
        
        # 偏导归零
        optimizer1.zero_grad()
        
        # 反向传播 
        loss.backward()
        
        # 更新参数        
        optimizer1.step()
          
        total_loss += loss.item()
        total_train_correct += get_num_correct(preds, labels)
    
    print("epoch:", epoch, 
          "correct times:", total_train_correct,
          "training accuracy:", "%.3f" %(total_train_correct/len(train_set)*100), "%", 
          "total_loss:", "%.3f" %total_loss)
    
time_end1 = time.time()

epoch: 0 correct times: 10084 training accuracy: 20.168 % total_loss: 541.742
epoch: 1 correct times: 17475 training accuracy: 34.950 % total_loss: 446.936
epoch: 2 correct times: 21282 training accuracy: 42.564 % total_loss: 397.087
epoch: 3 correct times: 23333 training accuracy: 46.666 % total_loss: 371.272
epoch: 4 correct times: 24855 training accuracy: 49.710 % total_loss: 351.193


<br>

<font color=black size=3 face=雅黑>　　请注意，由于本例中神经网络较小，数据集也较简单，所以使用 GPU 的加速效果不是特别显著，在复杂的任务中，GPU 将取得更好的加速效果。

<br>

## 2. PyTorch 与自动求导

<br>

<font color=black size=3 face=雅黑>　　在上次实验中，我们已经知道 PyTorch 有自动求导功能，通过 "loss.backward()" 可以很方便的实现神经网络的反向传播。下面我们就来进一步了解这个功能。

<code>
%%html
<img src = "https://gimg2.baidu.com/image_search/src=http%3A%2F%2Fpic3.zhimg.com%2Fv2-58b9712696a499fd9b01380a9926a3b3_1200x500.jpg&refer=http%3A%2F%2Fpic3.zhimg.com&app=2002&size=f9999,10000&q=a80&n=0&g=0n&fmt=jpeg?sec=1639062767&t=d95e4e63797c3b1a62afcab6c1fa2c0a", width=45%>
<br>

<font color=black size=3 face=雅黑>　　Tensor 是 PyTorch 最基础的数据类型。上次实验中，我们介绍了 tensor 的三个基本属性: shape, dtype 和 device。本次实验我们将学习三个与自动求导相关的 tensor 属性，分别是 requires_grad, grad 和 grad_fn。
    
<font color=black size=3 face=雅黑>　　其中，requires_grad 用于说明当前张量是否计算梯度信息（requires_grad=True 时保留）。对于那些需要计算梯度的 tensor，PyTorch 会存储他们的相关计算和梯度信息，这将造成额外的内存消耗。为了优化内存使用，不做特殊说明时，创建一个 tensor 默认是不需要梯度的（即 requires_grad 默认为 False）。
    
<font color=black size=3 face=雅黑>　　grad 属性对应张量的偏导。在 requires_grad=False 的情况下，grad 不会发生改变，其值等于 None 或之前已经计算过的 grad。
    
<font color=black size=3 face=雅黑>　　grad_fn 属性记录了得到这个 tensor 所进行的操作，例如加法、乘法运算等。
    
<br>

In [7]:
# 创建 tensor t1
t1 = torch.tensor([1,2,3], dtype=torch.float32)
print("t1: ", t1)

# 上次实验学习过的三种基本属性
print("t1.shape: ", t1.shape)  # 形状
print("t1.dtype: ", t1.dtype)  # 数据类型
print("t1.device: ", t1.device)  # 默认为 cpu
print("")

# 与自动求导相关的三种属性
print("t1.requires_grad: ", t1.requires_grad)  # 用于说明当前张量是否需要在计算中保留对应的梯度信息（默认 False）    
print("t1.grad: ", t1.grad)  # 偏导
print("t1.grad_fn: ", t1.grad_fn)  # 得到这个 tensor 进行的操作
print("")

# 基于 t1 计算得到张量 t2，其 requires_grad 属性默认与 t1 一致
t2 = t1*2
print("t2.requires_grad: ", t2.requires_grad)
print("t2.grad: ", t2.grad)  
print("t2.grad_fn: ", t2.grad_fn)  

t1:  tensor([1., 2., 3.])
t1.shape:  torch.Size([3])
t1.dtype:  torch.float32
t1.device:  cpu

t1.requires_grad:  False
t1.grad:  None
t1.grad_fn:  None

t2.requires_grad:  False
t2.grad:  None
t2.grad_fn:  None


In [8]:
# 因为 t2.requires_grad=False，此时使用如下代码对 t2[0] 反向求导会报错
t2[0].backward()

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

<br>

<font color=black size=3 face=雅黑>下面我们重新创建 t1，使其属性 requires_grad=True，观察以下代码的输出。
    
<br>

In [9]:
# 创建 tensor t1
t1 = torch.tensor([1,2,3], dtype=torch.float32, requires_grad=True)
print("t1: ", t1)

# 查看与自动求导相关的三种属性
print("t1.requires_grad: ", t1.requires_grad)  # 用于说明当前张量是否需要在计算中保留对应的梯度信息（默认 False）    
print("t1.grad: ", t1.grad)  # 偏导
print("t1.grad_fn: ", t1.grad_fn)  # 得到这个 tensor 进行的操作
print("")

# 基于 t1 计算得到张量 t2，其 requires_grad 属性默认与 t1 一致
t2 = t1*2
print("t2.requires_grad: ", t2.requires_grad)
print("t2.grad: ", t2.grad)  
print("t2.grad_fn: ", t2.grad_fn) 

t1:  tensor([1., 2., 3.], requires_grad=True)
t1.requires_grad:  True
t1.grad:  None
t1.grad_fn:  None

t2.requires_grad:  True
t2.grad:  None
t2.grad_fn:  <MulBackward0 object at 0x0000027AD10FD7E0>


C:\Users\trash\AppData\Local\Temp\ipykernel_11028\1645564063.py:14: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\build\aten\src\ATen/core/TensorBody.h:494.)
  print("t2.grad: ", t2.grad)


<br>

<font color=black size=3 face=雅黑>　　可以看到 t2 的 requires_grad 也为 True，且 t2.grad_fn 中记录了得到 t2 进行的操作 ("Mul" 表示乘法)。下面我们来看看自动求导是如何做的。
    
<br>

In [10]:
t2[0].backward()  # 我们对 t2[0] 反向求导

In [11]:
print("t1.grad: ", t1.grad)
print("t1.grad_fn: ", t1.grad_fn)

t1.grad:  tensor([2., 0., 0.])
t1.grad_fn:  None


<br>

<font color=black size=3 face=雅黑>　　下面我们再来看一个更接近实际情况的例子。发生在一个全连接层中**单个**神经元上的运算如下所示，其中 $a$ 表示输入激活值, $w$ 表示该神经元的权重，$b$ 表示偏置。该神经元上的线性计算结果 $z = w.T * a + b$，$z$ 是一个标量。
    
<br>

In [12]:
a = torch.tensor([[1.],[2.]], requires_grad=True)
w = torch.tensor([[3.],[4.]], requires_grad=True)
b = torch.tensor(6., requires_grad=True)

In [13]:
z = torch.mm(w.t(), a) + b
print(z)

tensor([[17.]], grad_fn=<AddBackward0>)


In [14]:
z.backward()

print("a.grad: ", a.grad)
print("w.grad: ", w.grad)
print("b.grad: ", b.grad)

a.grad:  tensor([[3.],
        [4.]])
w.grad:  tensor([[1.],
        [2.]])
b.grad:  tensor(1.)


<br>

<font color=black size=3 face=雅黑>我们可以通过如下方法 (requires_grad_) 来随时修改一个 tensor 的 requires_grad 属性。
    
<br>

In [15]:
a = torch.tensor([[1.],[2.]], requires_grad=True)
w = torch.tensor([[10.],[20.]], requires_grad=True)
b = torch.tensor(6., requires_grad=True)

In [16]:
a.requires_grad_(False)  # 将张量 a 的 requires_grad 属性重置为 False

z = torch.mm(w.t(), a) + b
print(z)

tensor([[56.]], grad_fn=<AddBackward0>)


In [17]:
z.backward()

print("a.grad: ", a.grad)  # None
print("w.grad: ", w.grad)
print("b.grad: ", b.grad)

a.grad:  None
w.grad:  tensor([[1.],
        [2.]])
b.grad:  tensor(1.)


<br>

<font color=black size=3 face=雅黑>在一个真实的神经网络中，我们可以很方便的查看每层参数的导数。首先，我们取出一个 batch 的样本，传递给 network，并计算损失。
    
<br>

In [18]:
network = Network()
batch = next(iter(train_loader))
images, labels = batch

preds = network(images)
loss = loss_func(preds, labels)
loss.item()

2.297414779663086

<br>

<font color=black size=3 face=雅黑>　　现在我们有了 loss，下一步可以使用 loss.backward() 计算偏导，PyTorch 会自动帮我们做相关的计算。在调用 loss.backward() 前，我们先检查一下第一个卷积层 conv1。

<br>

In [19]:
# 查看 conv1 信息
print("network.conv1: \n", network.conv1, "\n")

# 查看 conv1 参数
print("network.conv1.weight.data.shape: \n", network.conv1.weight.data.shape, "\n")
print("network.conv1.weight.data: \n", network.conv1.weight.data, "\n")

# 查看偏导用 network.conv1.weight.grad
# 运行以下代码，会发现输出为 None，说明目前还没有梯度
print("network.conv1.weight.grad: ", network.conv1.weight.grad)

network.conv1: 
 Conv2d(3, 6, kernel_size=(5, 5), stride=(1, 1)) 

network.conv1.weight.data.shape: 
 torch.Size([6, 3, 5, 5]) 

network.conv1.weight.data: 
 tensor([[[[ 0.0834,  0.0470,  0.0313,  0.1124,  0.1128],
          [-0.0263, -0.0112, -0.0252,  0.1154,  0.0578],
          [ 0.0069,  0.0242, -0.0389, -0.0097,  0.0534],
          [ 0.0009, -0.0455, -0.0036,  0.0217, -0.0668],
          [-0.0964, -0.0345, -0.1102,  0.0477,  0.0123]],

         [[-0.0729, -0.0515, -0.0496, -0.0931, -0.0348],
          [ 0.0977,  0.0042,  0.0085, -0.0824, -0.0562],
          [-0.0559,  0.0758,  0.0362, -0.1111,  0.0047],
          [-0.0112, -0.0091,  0.1026,  0.0867, -0.0328],
          [ 0.0364, -0.0142, -0.0120, -0.0474,  0.0830]],

         [[-0.0256,  0.1074,  0.0926, -0.0579,  0.0460],
          [-0.0159,  0.1077, -0.0350,  0.0012, -0.0124],
          [-0.0198,  0.0950,  0.1115, -0.0493,  0.1001],
          [-0.0155,  0.0177,  0.0573,  0.0034, -0.1064],
          [-0.0403, -0.0884,  0.0580, -0

In [20]:
# 现在运行反向函数
loss.backward()

In [21]:
# 再来看一下，会发现偏导被计算出来了，它是一个四维张量，其维度与 conv1 的权重张量相同
print("network.conv1.weight.grad.shape: \n", network.conv1.weight.grad.shape, "\n")
print("network.conv1.weight.grad: \n", network.conv1.weight.grad)

network.conv1.weight.grad.shape: 
 torch.Size([6, 3, 5, 5]) 

network.conv1.weight.grad: 
 tensor([[[[ 4.4885e-04,  3.5285e-04, -1.0256e-05, -2.1252e-04, -2.9670e-04],
          [ 3.0956e-04,  1.0949e-04, -5.0672e-04, -8.3907e-04, -7.4167e-04],
          [ 4.1967e-05, -2.1486e-04, -7.5391e-04, -1.0894e-03, -1.0476e-03],
          [-4.2884e-04, -5.6281e-04, -5.0663e-04, -8.5659e-04, -9.7525e-04],
          [-6.7171e-04, -8.3488e-04, -7.5286e-04, -8.1324e-04, -8.4601e-04]],

         [[ 8.0432e-04,  6.6136e-04,  2.5378e-04,  1.0171e-04,  8.2218e-05],
          [ 6.1116e-04,  3.3927e-04, -2.7751e-04, -4.1121e-04, -2.1052e-04],
          [ 2.5804e-04, -2.6605e-05, -4.8155e-04, -6.2985e-04, -5.2540e-04],
          [-2.0938e-04, -3.0224e-04, -7.2746e-05, -3.6144e-04, -4.8900e-04],
          [-4.3304e-04, -5.5854e-04, -3.6619e-04, -4.3236e-04, -4.8273e-04]],

         [[ 5.0961e-04,  4.9154e-04,  2.3613e-04,  1.6242e-04,  1.0289e-04],
          [ 3.7112e-04,  2.0783e-04, -2.6010e-04, -3.4549e